<a href="https://colab.research.google.com/github/tanveer-builds/LLM_from_Scratch/blob/main/fine_tunning/Instruction_Fine_Tunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Downloading the DataSet

In [4]:
import json
import os
import urllib
import ssl

def download_and_load_file(file_path, url):
  ssl_context = ssl.create_default_context()
  ssl_context.check_hostname = False
  ssl_context.verify_mode = ssl.CERT_NONE

  if not os.path.exists(file_path):
    with urllib.request.urlopen(url, context=ssl_context) as response:
      text_data = response.read().decode("utf-8")

    with open(file_path,"w", encoding="utf-8") as file:
      file.write(text_data)

  else:
    with open(file_path, "r", encoding="utf-8") as file:
      text_data = file.read()

  with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

  return data

In [5]:
file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
len(data)

1100

In [6]:
data[50]

{'instruction': 'Identify the correct spelling of the following word.',
 'input': 'Ocassion',
 'output': "The correct spelling is 'Occasion.'"}

# Convert to Alpaca Format

In [7]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [8]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


# Spliting Data

In [9]:
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
val_portion = int(len(data)) - train_portion - test_portion

train_data = data[:train_portion]
test_data = data[train_portion:train_portion+test_portion]
val_data = data[train_portion + test_portion:]

In [10]:
len(train_data), len(val_data), len(test_data)

(935, 55, 110)

In [11]:
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
  def __init__(self, data, tokenizer):
    self.data = data

    self.encoded_texts = []
    for entry in data:
      instruction_plus_input = format_input(entry)
      response_text = f"\n\n### Response:\n{entry["output"]}"
      full_text = instruction_plus_input + response_text
      self.encoded_texts.append(
          tokenizer.encode(full_text)
      )

  def __getitem__(self, index):
    return self.encoded_texts[index]

  def __len__(self):
    return len(self.data)


In [12]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


In [13]:
def custom_collate_draft_1(
    batch,
    pad_token_id = 50256,
    device="cpu"
):

  batch_max_length = max(len(item)+1 for item in batch)

  input_lst = []

  for item in batch:
    new_item = item.copy()

    padded = (
        new_item + [pad_token_id] *
        (batch_max_length - len(new_item))
    )

    inputs = torch.tensor(padded[:-1])
    input_lst.append(inputs)

  inputs_tensor = torch.stack(input_lst).to(device)
  return inputs_tensor


In [14]:
inputs_1 = [0, 1, 2, 3, 4, 5]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)

print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4,     5],
        [    5,     6, 50256, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256, 50256]])


In [19]:
def custom_collate_draft_2(
    batch,
    pad_token_id = 50256,
    device="cpu"
):

  batch_max_length = max(len(item)+1 for item in batch)

  input_lst = []
  target_lst = []
  for item in batch:
    new_item = item.copy()

    padded = (
        new_item + [pad_token_id] *
        (batch_max_length - len(new_item))
    )

    inputs = torch.tensor(padded[:-1])
    targets = torch.tensor(padded[1:])
    input_lst.append(inputs)
    target_lst.append(targets)

  inputs_tensor = torch.stack(input_lst).to(device)
  targets_tensor = torch.stack(target_lst).to(device)
  return inputs_tensor, targets_tensor


In [21]:
print(custom_collate_draft_2(batch))

(tensor([[    0,     1,     2,     3,     4,     5],
        [    5,     6, 50256, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256, 50256]]), tensor([[    1,     2,     3,     4,     5, 50256],
        [    6, 50256, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256, 50256]]))


In [29]:
def custom_collate_fn(
    batch,
    pad_token_id = 50256,
    ignore_index = -100,
    allowed_max_length=None,
    device="cpu"
):

  batch_max_length = max(len(item)+1 for item in batch)

  input_lst = []
  target_lst = []
  for item in batch:
    new_item = item.copy()

    #we are adding the end of text token after that we pad with -100
    #on -100 loss will not be calculated
    padded_input = (
        new_item + [pad_token_id] *
        (batch_max_length - len(new_item))
    )

    padded_target = (
        new_item + [pad_token_id] + [ignore_index] *
        (batch_max_length - len(new_item) -1)
    )

    inputs = torch.tensor(padded_input[:-1])
    targets = torch.tensor(padded_target[1:])

    if allowed_max_length is not None:
      inputs = inputs[:allowed_max_length]
      targets = targets[:allowed_max_length]

    input_lst.append(inputs)
    target_lst.append(targets)

  inputs_tensor = torch.stack(input_lst).to(device)
  targets_tensor = torch.stack(target_lst).to(device)
  return inputs_tensor, targets_tensor


In [30]:
print(custom_collate_fn(batch))

(tensor([[    0,     1,     2,     3,     4,     5],
        [    5,     6, 50256, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256, 50256]]), tensor([[    1,     2,     3,     4,     5, 50256],
        [    6, 50256,  -100,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100,  -100]]))
